In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers accelerate bitsandbytes sentencepiece faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.4 MB/s eta 0:00:00


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is uterine electromyography?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(llm.device)

In [ ]:
outputs = llm.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.2
)

In [ ]:
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Uterine electromyography (EMG) is not a standard medical procedure or diagnostic tool. It's possible you might be confusing it with another related concept. Here are some clarifications:

1. **Electromyography (EMG)**: This is a test that measures the electrical activity of muscles and nerves. It can be used to diagnose various conditions affecting the muscles and nerves.

2. **Uterine Electromyography (UEMG)**: This term is less commonly used and might refer to the measurement of electrical activity in the uterus during pregnancy. However, this specific use is not widely recognized or practiced in clinical settings.

If you're referring to UEMG, it could be an atypical or specialized application. In general, EMG tests are performed on muscles and nerves outside the uterus. If you have a specific context or source where you encountered this term, it would be helpful to provide more details for a more accurate explanation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [ ]:

PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"
EVALUATION_PATH = PROJECT_PATH / "evaluation"

In [ ]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def retrieve_candidates(
    query,
    model,
    index,
    metadata,
    k=20
):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "faiss_rank": rank,
            "faiss_score": float(
                scores[0][rank - 1]
            ),
            "index": int(idx),
            "paper": metadata.iloc[idx]["paper"],
            "chunk_id": metadata.iloc[idx]["chunk_id"],
            "text": metadata.iloc[idx]["text"]
        })

    return results

In [ ]:
def retrieve_and_rerank(
    query,
    embedding_model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
):

    # Stage 1: FAISS retrieval
    candidates = retrieve_candidates(
        query,
        embedding_model,
        index,
        metadata,
        k=candidate_k
    )

    # Stage 2: Cross-Encoder reranking
    pairs = [
        [query, result["text"]]
        for result in candidates
    ]

    scores = reranker.predict(pairs)

    for result, score in zip(
        candidates,
        scores
    ):
        result["rerank_score"] = float(score)

    # Sort
    candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return candidates[:final_k]

In [ ]:
query = (
    "What are the characteristics "
    "of uterine EMG signals?"
)

results = retrieve_and_rerank(
    query,
    model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
)

In [ ]:
for i, result in enumerate(results, start=1):

    print(
        f"\n--- Chunk {i} ---"
    )

    print(
        "Paper:",
        result["paper"]
    )

    print(
        "Chunk:",
        result["chunk_id"]
    )

    print(
        result["text"][:500]
    )


--- Chunk 1 ---
Paper: paper2
Chunk: 25
home environments.
2.4. Signal Characteristics and Typical EHG Profiles
Uterine EMG signals have distinctive characteristics in both the time and frequency do-
mains. Key features include amplitude, frequency content, temporal pattern, and propagation.
Amplitude: The EHG is a low-amplitude signal. During uterine contractions, the peak-
to-peak amplitude typically ranges from a few tens of microvolts up to about 1 millivolt on
the abdominal surface. Studies report that the higher-frequency “fast”

--- Chunk 2 ---
Paper: paper2
Chunk: 55
to enhance our ability to monitor, predict, and manage labor through the lens of uterine
electrophysiology.
3.4. Characteristics of Uterine EMG Signals (Cellular, Myometrial, and Abdominal Levels)
Uterine EMG manifests differently across biological scales. At the cellular level, sin-
gle myocytes show rhythmic slow depolarizations with superimposed action-potential
spikes recordable by microelectrodes; near term, 

In [ ]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(
        results,
        start=1
    ):

        context_parts.append(
            f"""
[Source {i}]
Paper: {result["paper"]}
Chunk: {result["chunk_id"]}

{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [ ]:
context = build_context(results)

In [ ]:
print(context)


[Source 1]
Paper: paper2
Chunk: 25

home environments.
2.4. Signal Characteristics and Typical EHG Profiles
Uterine EMG signals have distinctive characteristics in both the time and frequency do-
mains. Key features include amplitude, frequency content, temporal pattern, and propagation.
Amplitude: The EHG is a low-amplitude signal. During uterine contractions, the peak-
to-peak amplitude typically ranges from a few tens of microvolts up to about 1 millivolt on
the abdominal surface. Studies report that the higher-frequency “fast” component of EHG
usually has amplitude under 1 mV, whereas the low-frequency “slow” baseline shift can
reach several millivolts [10]. As labor progresses, the EHG amplitude tends to increase due
to stronger and more synchronized contractions. However, absolute amplitude can vary
widely between patients and with electrode placement, so amplitude alone is not a specific
indicator of labor unless normalized within-subject.


[Source 2]
Paper: paper2
Chunk: 55



In [ ]:
def build_prompt(query, context):

    prompt = f"""
You are a scientific research assistant specializing
in uterine electromyography (EMG/EHG).

Answer the user's question using ONLY the information
provided in the context below.

If the context does not contain enough information to
answer the question, say:

"I don't have enough information in the retrieved
research papers to answer this question."

Do not invent facts.

Cite the relevant sources using [Source N].

Context:
----------------
{context}
----------------

Question:
{query}

Answer:
"""

    return prompt

In [ ]:
prompt = build_prompt(
    query,
    context
)

print(prompt)


You are a scientific research assistant specializing
in uterine electromyography (EMG/EHG).

Answer the user's question using ONLY the information
provided in the context below.

If the context does not contain enough information to
answer the question, say:

"I don't have enough information in the retrieved
research papers to answer this question."

Do not invent facts.

Cite the relevant sources using [Source N].

Context:
----------------

[Source 1]
Paper: paper2
Chunk: 25

home environments.
2.4. Signal Characteristics and Typical EHG Profiles
Uterine EMG signals have distinctive characteristics in both the time and frequency do-
mains. Key features include amplitude, frequency content, temporal pattern, and propagation.
Amplitude: The EHG is a low-amplitude signal. During uterine contractions, the peak-
to-peak amplitude typically ranges from a few tens of microvolts up to about 1 millivolt on
the abdominal surface. Studies report that the higher-frequency “fast” component of EH

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a scientific research assistant. "
            "Answer questions using only the provided "
            "research context."
        )
    },
    {
        "role": "user",
        "content": prompt
    }
]

In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(llm.device)

In [ ]:
outputs = llm.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.2,
    do_sample=True
)

In [ ]:
answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

The characteristics of uterine EMG signals include:

- Amplitude: Typically ranges from a few tens of microvolts up to about 1 millivolt on the abdominal surface. Higher-frequency "fast" components usually have amplitudes under 1 mV, while low-frequency "slow" baseline shifts can reach several millivolts. The amplitude increases during labor due to stronger and more synchronized contractions.
- Frequency Content: Manifests as slow baseline shifts and fast-wave components up to ~1 Hz. 
- Propagation: Reflects the coordinated excitability of smooth muscle tissue.
- Origin: Derived from smooth-muscle action potentials, propagates via a coupled uterine syncytium, and manifests as low-frequency bursts correlating with contractions.
- Acquisition: Feasible with surface electrodes and specialized amplifiers.
- Processing: Extracted features such as peak frequency, which rises as labor approaches, or signal entropy, which falls as contractions become more organized.


In [ ]:
test_questions = [

    "What are the characteristics of uterine EMG signals?",

    "How is uterine EMG related to uterine contractions?",

    "What signal processing methods are commonly used for uterine EMG?",

    "How can uterine EMG be used for prediction of preterm labor?",

    "What are the differences between uterine EMG and skeletal muscle EMG?"
]

We are going to test the pipeline now

In [ ]:
import torch
import pandas as pd
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"

In [ ]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
def retrieve_candidates(
    query,
    model,
    index,
    metadata,
    k=20
):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(
        indices[0],
        start=1
    ):

        results.append({
            "faiss_rank": rank,
            "faiss_score": float(
                scores[0][rank - 1]
            ),
            "index": int(idx),
            "paper": metadata.iloc[idx]["paper"],
            "chunk_id": metadata.iloc[idx]["chunk_id"],
            "text": metadata.iloc[idx]["text"]
        })

    return results

In [ ]:
def retrieve_and_rerank(
    query,
    embedding_model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
):

    candidates = retrieve_candidates(
        query,
        embedding_model,
        index,
        metadata,
        k=candidate_k
    )

    pairs = [
        [query, result["text"]]
        for result in candidates
    ]

    scores = reranker.predict(pairs)

    for result, score in zip(
        candidates,
        scores
    ):
        result["rerank_score"] = float(score)

    candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    for rank, result in enumerate(
        candidates,
        start=1
    ):
        result["rerank_rank"] = rank

    return candidates[:final_k]

In [ ]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(
        results,
        start=1
    ):

        context_parts.append(
            f"""
[Source {i}]
Paper: {result["paper"]}
Chunk: {result["chunk_id"]}

{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [ ]:
def build_prompt(query, context):

    prompt = f"""
You are a scientific research assistant
specializing in uterine electromyography (EMG/EHG).

Answer the user's question using ONLY the information
provided in the research context below.

Do not use outside knowledge.

Do not invent facts.

If the retrieved context does not contain enough
information to answer the question, say:

"I don't have enough information in the retrieved
research papers to answer this question."

Cite the sources you use with [Source N].

Research Context:
-------------------------
{context}
-------------------------

Question:
{query}

Answer:
"""

    return prompt

In [ ]:
def rag_answer(
    query,
    candidate_k=20,
    final_k=5,
    max_new_tokens=300
):

    # ==========================================
    # STEP 1: RETRIEVE + RERANK
    # ==========================================

    results = retrieve_and_rerank(
        query=query,
        embedding_model=embedding_model,
        index=index,
        metadata=metadata,
        reranker=reranker,
        candidate_k=candidate_k,
        final_k=final_k
    )


    # ==========================================
    # STEP 2: BUILD CONTEXT
    # ==========================================

    context = build_context(results)


    # ==========================================
    # STEP 3: BUILD PROMPT
    # ==========================================

    prompt = build_prompt(
        query,
        context
    )


    # ==========================================
    # STEP 4: CREATE CHAT INPUT
    # ==========================================

    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific research assistant. "
                "Answer questions using only the provided "
                "research context."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # ==========================================
    # STEP 5: TOKENIZE
    # ==========================================

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm.device)


    # ==========================================
    # STEP 6: GENERATE ANSWER
    # ==========================================

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True
        )


    # ==========================================
    # STEP 7: DECODE ONLY NEW TOKENS
    # ==========================================

    answer = tokenizer.decode(
        outputs[0][
            inputs["input_ids"].shape[1]:
        ],
        skip_special_tokens=True
    )


    # ==========================================
    # STEP 8: RETURN EVERYTHING
    # ==========================================

    return {
        "query": query,
        "answer": answer,
        "sources": results
    }

In [ ]:
result = rag_answer(
    "What are the characteristics of uterine EMG signals?"
)

In [ ]:
print(result["answer"])

The characteristics of uterine EMG signals include:

1. Amplitude: The EHG is a low-amplitude signal. During uterine contractions, the peak-to-peak amplitude typically ranges from a few tens of microvolts up to about 1 millivolt on the abdominal surface. Higher-frequency "fast" components usually have amplitudes under 1 mV, while low-frequency "slow" baseline shifts can reach several millivolts. Absolute amplitude can vary widely between patients and with electrode placement.

2. Frequency Content: Uterine EMG signals exhibit a composite of slow baseline shifts and fast-wave components up to ~1 Hz. 

3. Temporal Pattern: The signals show a pattern of slow synchronous, calcium-mediated depolarizations reflecting the coordinated excitability of smooth muscle tissue.

4. Propagation: Uterine EMG signals originate from smooth-muscle action potentials, propagate via a coupled uterine syncytium, and manifest as low-frequency bursts correlating with contractions.

5. Signal Processing: Extrac

In [ ]:
for i, source in enumerate(
    result["sources"],
    start=1
):

    print(
        f"\nSource {i}"
    )

    print(
        "Paper:",
        source["paper"]
    )

    print(
        "Chunk:",
        source["chunk_id"]
    )

    print(
        "Rerank score:",
        source["rerank_score"]
    )


Source 1
Paper: paper2
Chunk: 25
Rerank score: 8.740561485290527

Source 2
Paper: paper2
Chunk: 55
Rerank score: 8.26572036743164

Source 3
Paper: paper2
Chunk: 44
Rerank score: 6.40878963470459

Source 4
Paper: paper2
Chunk: 54
Rerank score: 5.959493160247803

Source 5
Paper: paper2
Chunk: 87
Rerank score: 5.5933732986450195


In [ ]:
test_questions = [

    "What are the characteristics of uterine EMG signals?",

    "How is uterine EMG related to uterine contractions?",

    "What signal processing methods are commonly used for uterine EMG?",

    "How can uterine EMG be used for prediction of preterm labor?",

    "What are the differences between uterine EMG and skeletal muscle EMG?"
]

In [ ]:
for question in test_questions:

    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    result = rag_answer(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")

    for i, source in enumerate(
        result["sources"],
        start=1
    ):

        print(
            f"[Source {i}] "
            f"{source['paper']} "
            f"- Chunk {source['chunk_id']}"
        )

    print()

QUESTION:
What are the characteristics of uterine EMG signals?

ANSWER:
The characteristics of uterine EMG signals include:

1. Amplitude: The EHG is a low-amplitude signal. During uterine contractions, the peak-to-peak amplitude typically ranges from a few tens of microvolts up to about 1 millivolt on the abdominal surface. Higher-frequency "fast" components usually have amplitudes under 1 mV, while low-frequency "slow" baseline shifts can reach several millivolts. The amplitude increases during labor due to stronger and more synchronized contractions.

2. Frequency Content: Uterine EMG signals exhibit a slow, synchronous, calcium-mediated depolarization characteristic. They show a composite of slow baseline shifts and fast-wave components up to ~1 Hz. 

3. Time Pattern: The signals are recorded with surface electrodes and specialized amplifiers. They typically show a composite of slow baseline shifts and fast-wave components up to ~1 Hz.

4. Propagation: Uterine EMG signals originate

In [ ]:
for i, source in enumerate(
    result["sources"],
    start=1
):

    print(f"\n{'='*80}")
    print(f"[Source {i}]")
    print("Paper:", source["paper"])
    print("Chunk:", source["chunk_id"])
    print("Rerank score:", source["rerank_score"])
    print("\nText:")
    print(source["text"])


[Source 1]
Paper: paper2
Chunk: 45
Rerank score: 7.682560443878174

Text:
and paracrine influences. Uterine action potentials are calcium-based plateau spikes with
long durations, supporting sustained contractions. Consequently, uterine EMG signals
are low-frequency, long-duration bursts rather than the brief spikes typical of skeletal-
muscle EMG.
Frequency content: Typical skeletal-muscle EMG (from a biceps or leg muscle, for
example) has significant power in the 50–150 Hz range due to short-duration action poten-
tials (2–5 ms) and fast conduction velocities. Uterine muscle action potentials last much
longer (50–1000 ms, including plateaus) and propagate slowly, yielding frequency content
mainly under 5 Hz. This low-frequency nature demands specialized amplifiers and filters,
as conventional EMG systems might otherwise suppress these frequencies as “noise” [10].
Propagation and recruitment: Whereas skeletal muscles consist of independently
firing motor units, the uterus behaves as 

In [ ]:
def ask_rag(question):

    result = rag_answer(question)

    print("\nAnswer:")
    print(result["answer"])

    print("\nSources:")

    for i, source in enumerate(
        result["sources"],
        start=1
    ):

        print(
            f"[{i}] {source['paper']} "
            f"(Chunk {source['chunk_id']})"
        )

In [ ]:
ask_rag(
    "How can uterine EMG be used for prediction of preterm labor?"
)


Answer:
Uterine EMG (Electromyography) can be used for the prediction of preterm labor through various EHG (Electrohysterography) parameters. Changes in these parameters reflect physiological phenomena leading to delivery. Studies have shown that EHG features are "dynamic" and change throughout pregnancy. As labor approaches, uterine electrical activity becomes more intense and synchronized. Different EMG parameters can indicate myometrial properties that distinguish between true and 'false' labor contractions in both term and preterm pregnancies. Uterine EMG can help identify patients in true labor better than other methods currently used in clinics. Additionally, EHG is non-invasive, requiring no special facilities or equipment, making it a cost-effective method.

Sources:
[1] paper2 (Chunk 17)
[2] paper7 (Chunk 42)
[3] paper9 (Chunk 44)
[4] paper1 (Chunk 194)
[5] paper7 (Chunk 35)
